In [66]:
import json
from pathlib import Path

racine = Path.cwd().parent.parent # fonctionne avec notebook
chemin = racine/"projet"/"donnees_experimentales"
with open(chemin / "development.json", "r", encoding="utf-8") as fichier:
    development = json.load(fichier)

# Extraire la liste des articles
articles = development["essays"]

print("Nombre d'articles :", len(articles))

print("\nClés du premier article :")
print(articles[0].keys())

print("\nTitre :")
print(articles[0]["title"])

print("\nIdentifiant :")
print(articles[0]["essay_id"])

print("\nLongueur du texte HTML :")
print(len(articles[0]["raw_text"]))

Nombre d'articles : 360

Clés du premier article :
dict_keys(['dossier_ids', 'essay_id', 'language_code', 'raw_text', 'source_author', 'source_category', 'source_published_at', 'source_updated_at', 'title', 'url'])

Titre :
Un premier pas législatif pour officialiser la formation à distance au Cameroun. -

Identifiant :
mysql_article_10024

Longueur du texte HTML :
4346


In [67]:
from bs4 import BeautifulSoup
import re

def nettoyer_html(html):
    soup = BeautifulSoup(html, "html.parser")

    # Remplacer les balises structurantes par des séparateurs
    for balise in soup.find_all([
        "p", "div", "br", "h1", "h2", "h3",
        "h4", "h5", "h6", "li", "blockquote"
    ]):
        balise.append("\n")

    texte = soup.get_text()

    # Nettoyer les espaces
    texte = re.sub(r"[ \t]+", " ", texte)
    texte = re.sub(r"\n\s*\n+", "\n\n", texte)

    return texte.strip()

In [68]:
texte_test = nettoyer_html(articles[0]["raw_text"])

print(texte_test[:3000])



En mars dernier, les parlementaires camerounais ont passé au peigne fin la loi d’orientation de l’enseignement supérieur dans leur pays. Cette nouvelle loi vise à asseoir sur une base législative le système camerounais de l’enseignement supérieur qui, jusque là, n’était régi que par des disposition réglementaires. 
Dans le texte de la loi, l’actuel ministre de l’Enseignement supérieur -- un homme qui imprime à la vie universitaire un rythme nouveau et dynamique relativement aux défis posés par les nouvelles technologies et leurs rapports avec la formation à distance -- détermine les orientations fondamentales des institutions universitaires et fixe les règles générales d’organisation, de fonctionnement, de gestion, de financement et de contrôle des établissements universitaires publics et privés.
La FAD au service des populations et des citoyensRespectueux des clauses établies avec la Banque Mondiale et le FMI, suivant en cela le rapport de l’UNESCO déjà présenté dans Thot, le gouverne

In [6]:
dossiers = development["dossiers"]
print("Nombre de dossiers :", len(dossiers))

Nombre de dossiers : 20


In [11]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://seshat.cursus.edu/fr/embed/analysis/10024"

response = requests.get(url, timeout=20)

print("Code HTTP :", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

texte = soup.get_text(separator=" ", strip=True)

print(texte)

Code HTTP : 200
SESHAT AI Seshat AI Analyse Tension principale Tradition vs Changement Tension secondaire Autonomie vs Interdépendance Questions socratiques Comment la formation à distance peut-elle être pleinement reconnue et intégrée dans le système éducatif camerounais ? Quels sont les obstacles institutionnels majeurs à la modernisation de l'enseignement supérieur au Cameroun ? Comment assurer une meilleure coordination entre les ministères concernés pour développer efficacement la formation à distance ? Essayer Seshat AI Seshat AI est un assistant socratique contrôlé par l'enseignant, conçu pour vous aider à enseigner l'esprit critique et à préparer vos cours plus rapidement, tout en préservant l'exigence intellectuelle.


In [12]:
resultat = re.search(
    r"Tension principale\s*(.*?)\s*"
    r"Tension secondaire\s*(.*?)\s*"
    r"Questions socratiques",
    texte
)

if resultat:
    tension_primaire = resultat.group(1).strip()
    tension_secondaire = resultat.group(2).strip()

    print("Tension primaire :", tension_primaire)
    print("Tension secondaire :", tension_secondaire)
else:
    print("Annotations introuvables")

Tension primaire : Tradition vs Changement
Tension secondaire : Autonomie vs Interdépendance


### importer les données de development sur les articles

In [16]:
import pandas as pd

df = pd.DataFrame(articles)

print(df.shape)
print(df.columns)
print(df.head())

(360, 10)
Index(['dossier_ids', 'essay_id', 'language_code', 'raw_text', 'source_author',
       'source_category', 'source_published_at', 'source_updated_at', 'title',
       'url'],
      dtype='object')
  dossier_ids             essay_id language_code  \
0          []  mysql_article_10024            fr   
1          []  mysql_article_10097            fr   
2          []  mysql_article_10099            fr   
3          []  mysql_article_10129            fr   
4          []  mysql_article_10226            fr   

                                            raw_text  \
0  <p>En mars dernier, les parlementaires camerou...   
1  <p><strong>Dans la réalité, une université rép...   
2  <p><a href="http://www.syren.fr/syra-media" ti...   
3  <p>Initiative du Comité de Solidarité Tiers-Mo...   
4  <p>L’accord national interprofessionnel du 20 ...   

                 source_author source_category source_published_at  \
0  Louis Martin Onguene Essono       Activités          2001-05-17   
1   

### nettoyer les articles

In [17]:
from bs4 import BeautifulSoup

def nettoyer_html(texte):
    soup = BeautifulSoup(texte, "html.parser")
    return soup.get_text(separator=" ", strip=True)

df["clean_text"] = df["raw_text"].apply(nettoyer_html)

### charger les dossiers (correspond à la ligne éditoriale d'une série d'articles

In [48]:
dossiers = development["dossiers"]

df_dossiers = pd.DataFrame(dossiers)

print("Nombre de dossiers :", len(df_dossiers))
# print(df_dossiers.keys())
# print(df_dossiers.head())
print(df_dossiers["essay_ids"])

longueur = 0
for x in df_dossiers["essay_ids"]:
    longueur += len(x)
print(longueur)




Nombre de dossiers : 20
0     [mysql_article_34385, mysql_article_35449, mys...
1     [mysql_article_11316, mysql_article_11465, mys...
2     [mysql_article_20729, mysql_article_25999, mys...
3     [mysql_article_24648, mysql_article_27956, mys...
4     [mysql_article_36585, mysql_article_44534, mys...
5     [mysql_article_24720, mysql_article_27381, mys...
6     [mysql_article_34203, mysql_article_35185, mys...
7     [mysql_article_1833, mysql_article_26457, mysq...
8     [mysql_article_11564, mysql_article_27455, mys...
9     [mysql_article_27238, mysql_article_44377, mys...
10    [mysql_article_2553, mysql_article_35259, mysq...
11    [mysql_article_26209, mysql_article_26297, mys...
12    [mysql_article_25325, mysql_article_2553, mysq...
13    [mysql_article_25441, mysql_article_25669, mys...
14    [mysql_article_24848, mysql_article_35655, mys...
15    [mysql_article_27691, mysql_article_35654, mys...
16    [mysql_article_33143, mysql_article_33560, mys...
17    [mysql_article_205

### charger les tensions

In [20]:
import re

def obtenir_url_seshat(essay_id):
    resultat = re.search(r"(\d+)$", str(essay_id))

    if resultat:
        identifiant = resultat.group(1)
        return f"https://seshat.cursus.edu/fr/embed/analysis/{identifiant}"

    return None

In [21]:
for i in range(5):
    print(
        df.loc[i, "essay_id"],
        obtenir_url_seshat(df.loc[i, "essay_id"])
    )

mysql_article_10024 https://seshat.cursus.edu/fr/embed/analysis/10024
mysql_article_10097 https://seshat.cursus.edu/fr/embed/analysis/10097
mysql_article_10099 https://seshat.cursus.edu/fr/embed/analysis/10099
mysql_article_10129 https://seshat.cursus.edu/fr/embed/analysis/10129
mysql_article_10226 https://seshat.cursus.edu/fr/embed/analysis/10226


In [62]:
import requests
from bs4 import BeautifulSoup
import time

def recuperer_analyse_seshat(essay_id):

    url = obtenir_url_seshat(essay_id)

    if url is None:
        return None

    try:
        reponse = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        if reponse.status_code != 200:
            print("Erreur HTTP :", reponse.status_code, essay_id)
            return None

        soup = BeautifulSoup(reponse.text, "html.parser")

        texte = soup.get_text(separator="\n", strip=True)

        return texte

    except requests.RequestException as erreur:
        print("Erreur :", essay_id, erreur)
        return None

In [63]:
texte = recuperer_analyse_seshat("mysql_article_43986")

print(texte)

SESHAT AI
Seshat AI
Analyse
Tension principale
Rationalité vs Émotion
Tension secondaire
Objectivité vs Subjectivité
Questions socratiques
Comment la science-fiction influence-t-elle concrètement la recherche et l'innovation ?
Quels sont les avantages et limites de la prospective face à l'imprévu ?
En quoi l'imagination narrative complète-t-elle l'approche analytique dans la préparation au futur ?
Essayer Seshat AI
Seshat AI est un assistant socratique contrôlé par l'enseignant, conçu pour vous aider à enseigner l'esprit critique et à préparer vos cours plus rapidement, tout en préservant l'exigence intellectuelle.


In [61]:
print(texte[:5000])

SESHAT AI
Seshat AI
Analyse
Tension principale
Rationalité vs Émotion
Tension secondaire
Objectivité vs Subjectivité
Questions socratiques
Comment la science-fiction influence-t-elle concrètement la recherche et l'innovation ?
Quels sont les avantages et limites de la prospective face à l'imprévu ?
En quoi l'imagination narrative complète-t-elle l'approche analytique dans la préparation au futur ?
Essayer Seshat AI
Seshat AI est un assistant socratique contrôlé par l'enseignant, conçu pour vous aider à enseigner l'esprit critique et à préparer vos cours plus rapidement, tout en préservant l'exigence intellectuelle.


In [64]:
# extraction des tensions

resultat = re.search(
    r"Tension principale\s*(.*?)\s*"
    r"Tension secondaire\s*(.*?)\s*"
    r"Questions socratiques",
    texte
)

if resultat:
    tension_primaire = resultat.group(1).strip()
    tension_secondaire = resultat.group(2).strip()

    print("Tension primaire :", tension_primaire)
    print("Tension secondaire :", tension_secondaire)
else:
    print("Annotations introuvables")

Tension primaire : Rationalité vs Émotion
Tension secondaire : Objectivité vs Subjectivité
